# §7 concat diagnostics — C1⊕C3 with the C1⊕C1′ ensemble control

Thin notebook (§7 v5.2 "diagnostiche aggiuntive val-only"): does
SupCon (C3) capture activity information that CE (C1) misses, or does
any second encoder help the same way? Two 512-d feature
concatenations, the frozen §5.3 probe recipe **unchanged**
(`probe.linear_probe` on raw concatenated features — no per-block
rescaling, the linear head absorbs scale; declared §7), seed 42,
val-only:

- **C1 ⊕ C3** (epoch40, the phase-B selection) — the complementarity
  candidate;
- **C1 ⊕ C1′** (seed 42 ⊕ seed 43) — the control: pure ensemble
  diversity within the same loss family.

Reading, declared a priori: C1⊕C3 ≈ C1⊕C1′ → C3's contribution is
generic ensemble diversity, not CE/SupCon complementarity; C1⊕C3
above the control by more than the §0.5 ~2-pt band would be the
complementarity signal. Context: the error-overlap analysis found only
16/349 unstructured rescue windows, so the pre-declared expectation is
"no complementarity" — the number still goes in the report either
way (future-work sentence for joint CE+SupCon).

Alignment across caches is **asserted, never fixed**
(`diagnostics.concat_caches` blocks on any per-row trace/window/
antenna/label mismatch — wrong files, not a reorder to repair).

No staging (cached features only — CPU runtime is fine). No test
contact (§0.7). If the session runs on a later date, `git mv` this
template to the run date first (folder rule: date = when the session
ran); the executed copy is committed over it verbatim + README index
row + STATUS line, same commit.

## Environment setup

Diagnostics preamble: repo + Drive only, no data staging. Needs
read access to the cached features under `C1/`, `C3/` and `C1_s43/`
(Drive shortcuts — same requirement as the NCM/kNN session).

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"

cwd = Path.cwd().resolve()
if (cwd / "sharp_har").exists():
    REPO_DIR = cwd
elif (cwd.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent
elif (cwd.parent.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent.parent
else:
    REPO_DIR = Path("/content/sharp-har")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# After the clone, so the file exists on a fresh runtime.
!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, str(REPO_DIR))

from google.colab import drive
from sharp_har.utils import read_yaml

drive.mount("/content/drive")   # idempotent
CKPT_ROOT = Path(read_yaml(REPO_DIR / "configs" / "paths.yaml")["ckpt_root"])
print("Repo dir:", REPO_DIR)
print("Checkpoint root:", CKPT_ROOT)

## Imports

Recipes only — nothing is defined here (CLAUDE.md: no logic in
notebooks).

In [ ]:
import numpy as np

from sharp_har.diagnostics import concat_caches, fused_head_scores
from sharp_har.probe import linear_probe

## Run

Caches (all pre-existing, produced by the probe / phase-B / E1-tail
sessions): `C1/features_best_{train,val}.npz`,
`C3/features_epoch40_{train,val}.npz`,
`C1_s43/features_best_{train,val}.npz`. A missing cache SKIPs its pair
with a note (the C1⊕C1′ control un-SKIPs itself once
`04_cache_c1_s43_features.ipynb` has run).

Single-encoder references are the frozen recorded numbers (STATUS):
they are printed for the delta reading, not recomputed.

In [ ]:
SEED = 42  # §0 rule 5
D_EXPECTED = 512  # 256 + 256, printed as a sanity check (not tuned on)

# Frozen single-encoder records (val macro-F1, fused §1.3; STATUS 2026-07-17/18).
REFERENCE = {"C1-lin": 0.8835, "C3-lin (epoch40)": 0.8190}

PAIRS = {
    "C1(+)C3": (("C1", "best"), ("C3", "epoch40")),
    "C1(+)C1' [control]": (("C1", "best"), ("C1_s43", "best")),
}

results = {}
for name, specs in PAIRS.items():
    paths = {s: [CKPT_ROOT / folder / f"features_{stem}_{s}.npz" for folder, stem in specs]
             for s in ("train", "val")}
    missing = [p for ps in paths.values() for p in ps if not p.exists()]
    if missing:
        print(f"[{name}] SKIP - missing cache(s):")
        for p in missing:
            print(f"    {p}")
        if any("C1_s43" in str(p) for p in missing):
            print("    (run 04_cache_c1_s43_features.ipynb first - E1 tail, see STATUS)")
        continue

    tr = concat_caches(*paths["train"])
    va = concat_caches(*paths["val"])
    assert tr["features"].shape[1] == D_EXPECTED, tr["features"].shape

    res = linear_probe(tr, va, target="y", seed=SEED)
    scores = fused_head_scores(
        va["features"], va["y"], va["trace_id"], va["window_start"],
        res["weight"], res["bias"],
    )
    results[name] = {"val_macro_f1": res["best_val_macro_f1"],
                     "val_accuracy": scores["accuracy"],
                     "best_epoch": res["best_epoch"]}
    print(f"[{name}] val macro-F1 {res['best_val_macro_f1']:.4f}, "
          f"val accuracy {scores['accuracy']:.4f} (best probe epoch {res['best_epoch']})")

print()
for k, v in REFERENCE.items():
    print(f"reference {k}: val macro-F1 {v:.4f}")
for name, r in results.items():
    print(f"{name}: val macro-F1 {r['val_macro_f1']:.4f} "
          f"(vs C1-lin {r['val_macro_f1'] - REFERENCE['C1-lin']:+.4f})")
if len(results) == 2:
    ks = list(results)
    gap = results[ks[0]]["val_macro_f1"] - results[ks[1]]["val_macro_f1"]
    print(f"\nC1(+)C3 minus control: {gap:+.4f}  "
          "(complementarity claim needs > ~0.02, the §0.5 band; "
          "remember the 5-class val caveat - test never sees these heads)")

## Archiving

Download the executed copy and commit it verbatim over this file (or
under the actual run date, see the header) + README index row + STATUS
line, same commit. The probe heads trained here are diagnostic-only:
they are NOT persisted and never see the test set (§0.7 — no
pre-registered test row exists for the concat).